# Deep Learning Final Examination: Grid Search for TickNet-L
## CIFAR-10 & CIFAR-100 Benchmarking with SGD and Adam

This notebook executes the full 8-experiment Grid Search ablation study for the **TickNet-L v1** architecture on CIFAR-10 and CIFAR-100 as required by .

### 8 Experimental Configurations:
1. **CIFAR-10 | SGD lr=0.10** ()
2. **CIFAR-10 | SGD lr=0.15** ()
3. **CIFAR-10 | Adam lr=0.001** ()
4. **CIFAR-10 | Adam lr=0.0003** ()
5. **CIFAR-100 | SGD lr=0.10** ()
6. **CIFAR-100 | SGD lr=0.15** ()
7. **CIFAR-100 | Adam lr=0.001** ()
8. **CIFAR-100 | Adam lr=0.0003** ()


In [ ]:
# 1. Environment & GPU Preflight Check
import os
import torch

# Auto-clone repository if running in fresh Kaggle environment
if not os.path.exists("train_cifar.py"):
    !test -d /kaggle/working/TickNets/.git || git clone --depth 1 --branch feature/final-exam-model-l https://github.com/khuonglaptri3/TickNets.git /kaggle/working/TickNets
    %cd /kaggle/working/TickNets

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
!nvidia-smi


In [ ]:
# 2. Download and Verify Datasets
!python download_cifar.py --data-root /kaggle/working/data


## 3. Run CIFAR-10 Experiments (4 Configurations)

In [ ]:
# 3.1. CIFAR-10 SGD lr=0.10
!python train_cifar.py --config configs/final/cifar10_sgd_lr010.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar10_sgd_lr010


In [ ]:
# 3.2. CIFAR-10 SGD lr=0.15
!python train_cifar.py --config configs/final/cifar10_sgd_lr015.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar10_sgd_lr015


In [ ]:
# 3.3. CIFAR-10 Adam lr=0.001
!python train_cifar.py --config configs/final/cifar10_adam_lr0001.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar10_adam_lr0001


In [ ]:
# 3.4. CIFAR-10 Adam lr=0.0003
!python train_cifar.py --config configs/final/cifar10_adam_lr00003.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar10_adam_lr00003


## 4. Run CIFAR-100 Experiments (4 Configurations)

In [ ]:
# 4.1. CIFAR-100 SGD lr=0.10
!python train_cifar.py --config configs/final/cifar100_sgd_lr010.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar100_sgd_lr010


In [ ]:
# 4.2. CIFAR-100 SGD lr=0.15
!python train_cifar.py --config configs/final/cifar100_sgd_lr015.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar100_sgd_lr015


In [ ]:
# 4.3. CIFAR-100 Adam lr=0.001
!python train_cifar.py --config configs/final/cifar100_adam_lr0001.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar100_adam_lr0001


In [ ]:
# 4.4. CIFAR-100 Adam lr=0.0003
!python train_cifar.py --config configs/final/cifar100_adam_lr00003.json --data-root /kaggle/working/data --output-dir /kaggle/working/runs/cifar100_adam_lr00003


## 5. Master Comparison Table & Summary Report

In [ ]:
import json
from pathlib import Path
import pandas as pd

runs_dir = Path("/kaggle/working/runs")
records = []

configs = [
    "cifar10_sgd_lr010", "cifar10_sgd_lr015", "cifar10_adam_lr0001", "cifar10_adam_lr00003",
    "cifar100_sgd_lr010", "cifar100_sgd_lr015", "cifar100_adam_lr0001", "cifar100_adam_lr00003",
]

for exp_name in configs:
    p = runs_dir / exp_name
    metric_file = p / "test_metrics.json"
    cfg_file = p / "config.json"
    if metric_file.is_file() and cfg_file.is_file():
        metrics = json.loads(metric_file.read_text())
        cfg = json.loads(cfg_file.read_text())
        records.append({
            "Experiment": exp_name,
            "Dataset": cfg["dataset"].upper(),
            "Optimizer": cfg["optimizer"].upper(),
            "LR": cfg["learning_rate"],
            "Test Top-1 (%)": f"{metrics['top1']:.2f}%",
            "Test Loss": f"{metrics['loss']:.4f}",
            "Macro F1": f"{metrics['macro_f1']:.4f}",
            "Params": f"{cfg['learnable_parameters']:,}",
            "FLOPs (G)": f"{cfg['gflops_forward']:.4f}G",
        })

df = pd.DataFrame(records)
print("=== MASTER GRID SEARCH RESULTS ===")
display(df)
df.to_csv("/kaggle/working/runs/master_grid_search_summary.csv", index=False)
